In [2]:
# train.py - QLoRA fine-tuning for the Romanized Nepali rerun (identical for every model and seed).
# Run in a Kaggle notebook (GPU T4 x2, Internet on):
!pip -q install unsloth trl peft bitsandbytes datasets
#   then edit the CONFIG block below and run this file (%run train.py).
# Set GATE = True first: 50 steps, prints the checks, then stop and read the output.
import json, math, os, time
import torch

# ---------------- CONFIG (the only block you edit) ----------------
MODEL_ID = "Qwen/Qwen2.5-3B"     # google/gemma-2-2b | Qwen/Qwen2.5-3B | meta-llama/Llama-3.2-3B
SEED = 42                                # 42, 43, 44
DATA_DIR = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected"  # holds train.jsonl and validation.jsonl
OUT_DIR = "/kaggle/working/run"          # adapter, logs and training_meta.json are written here
GATE = False                              # True = 50-step check only
# ------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

ALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "
          "Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
ALPACA_NO_INPUT = ("Below is an instruction that describes a task. "
                   "Write a response that appropriately completes the request.\n\n"
                   "### Instruction:\n{instruction}\n\n### Response:\n")

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=512, dtype=torch.float16, load_in_4bit=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none", use_rslora=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"TRAINABLE PARAMETERS: {n_train:,} (counted on the 4-bit model; {n_all:,} total as stored)")

def fmt(ex):
    p = (ALPACA if ex["input"].strip() else ALPACA_NO_INPUT).format(**ex)
    return {"text": p + ex["output"] + tok.eos_token}

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                        "validation": f"{DATA_DIR}/validation.jsonl"})
data = data.map(fmt)

# --- sanity checks that must be read in the gate run ---
first = tok(data["train"][0]["text"])["input_ids"]
print("BOS id:", tok.bos_token_id, "| first token id:", first[0],
      "| starts with BOS:", (first[0] == tok.bos_token_id) if tok.bos_token_id is not None else "n/a (no BOS)")
print("Last 3 token ids of first example:", first[-3:], "| EOS id:", tok.eos_token_id,
      "(should end with exactly one EOS)")
print("Vocab size:", len(tok), "| uniform-guess loss ln(V) = %.2f" % math.log(len(tok)))

cfg = SFTConfig(
    output_dir=OUT_DIR, dataset_text_field="text", max_seq_length=512, packing=False,
    per_device_train_batch_size=2, gradient_accumulation_steps=4, per_device_eval_batch_size=2,
    num_train_epochs=3, max_steps=50 if GATE else -1,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=200, optim="adamw_8bit",
    fp16=True, bf16=False, seed=SEED, data_seed=SEED,
    eval_strategy="steps", eval_steps=100 if not GATE else 25,
    save_strategy="steps", save_steps=100 if not GATE else 25, save_total_limit=10,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, report_to="none")
# NOTE: in some TRL versions max_seq_length is called max_length. Rename it if you get an error.

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=data["train"],
                     eval_dataset=data["validation"], args=cfg)

step0 = trainer.evaluate()["eval_loss"]
print(f"STEP-0 EVAL LOSS: {step0:.3f}  (must be well below ln(V) above; the old Gemma run showed 22.6)")

t0 = time.time()
trainer.train()
secs = time.time() - t0
steps = trainer.state.global_step
print(f"SECONDS PER STEP: {secs/steps:.2f}")

trainer.model.save_pretrained(f"{OUT_DIR}/adapter")
tok.save_pretrained(f"{OUT_DIR}/adapter")
meta = {"model": MODEL_ID, "seed": SEED, "gate": GATE, "trainable_params": n_train,
        "effective_batch": cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps,
        "num_gpus_visible": torch.cuda.device_count(), "world_size": trainer.args.world_size,
        "global_steps": steps, "seconds_per_step": secs / steps, "step0_eval_loss": step0,
        "best_step": trainer.state.best_global_step, "best_eval_loss": trainer.state.best_metric,
        "log_history": trainer.state.log_history}
json.dump(meta, open(f"{OUT_DIR}/training_meta.json", "w"), indent=1)
print("DONE. Best step:", meta["best_step"], "best val loss:", meta["best_eval_loss"],
      "| steps:", steps, "| effective batch:", meta["effective_batch"], "| world size:", meta["world_size"])

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
TRAINABLE PARAMETERS: 59,867,136 (counted on the 4-bit model; 1,927,622,656 total as stored)


Generating train split: 0 examples [00:00, ? examples/s]

... (12 outputs left out)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/checkpoint-900/tokenizer_config.json.
Unsloth: Restored a

SECONDS PER STEP: 3.24


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/run/adapter/tokenizer_config.json.


DONE. Best step: 3100 best val loss: 1.350008249282837 | steps: 3375 | effective batch: 8 | world size: 1


In [6]:
%%writefile /kaggle/working/train_canonical.py
# eval.py v2.2 (adds scoring cap, early save of generations, per-item NLL) - one evaluation script for zero-shot and fine-tuned models (identical for every model).
# Run in a Kaggle notebook (GPU, Internet on):
!pip -q install unsloth peft bitsandbytes datasets sacrebleu rouge-score bert-score
# Edit CONFIG, then %run eval.py. Writes outputs.jsonl and metrics.json into OUT_DIR.
import json, math, os, random, re
import numpy as np
import torch

# ---------------- CONFIG ----------------
MODE = "finetuned"                       # "zeroshot" (bare instruction, base weights) or "finetuned"
MODEL_ID = "Qwen/Qwen2.5-3B"     # base model id (used for zeroshot)
ADAPTER_DIR = "/kaggle/input/datasets/pujanpant017/adaptaer-all/adapter_001/adapter_001"   # used when MODE == "finetuned"
DATA_FILE = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected/test.jsonl"
OUT_DIR = "/kaggle/working/eval_seed42"
N_ITEMS = None                           # None = all 1,000; set e.g. 500 only if the pilot says time is short
# ----------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
from unsloth import FastLanguageModel

ALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "
          "Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
ALPACA_NO_INPUT = ("Below is an instruction that describes a task. "
                   "Write a response that appropriately completes the request.\n\n"
                   "### Instruction:\n{instruction}\n\n### Response:\n")

def build_prompt(r):
    if MODE == "zeroshot":   # Rimal and Rimal: no system prompt, template or few-shot examples
        return r["instruction"] + ("\n" + r["input"] if r["input"].strip() else "")
    return (ALPACA if r["input"].strip() else ALPACA_NO_INPUT).format(**r)

rows = [json.loads(l) for l in open(DATA_FILE)]
if N_ITEMS:
    random.Random(42).shuffle(rows); rows = rows[:N_ITEMS]

load_name = ADAPTER_DIR if MODE == "finetuned" else MODEL_ID   # adapter on the 4-bit base, never merged
model, tok = FastLanguageModel.from_pretrained(model_name=load_name, max_seq_length=4096,
                                               dtype=torch.float16, load_in_4bit=True)
FastLanguageModel.for_inference(model)
print("model.max_seq_length:", getattr(model, "max_seq_length", "n/a"), flush=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

# ---- 1. greedy generation, identical for both stages ----
prompts = [build_prompt(r) for r in rows]
n_long_prompts = sum(len(tok(p, add_special_tokens=True)["input_ids"]) > 512 for p in prompts)
print("prompts longer than 512 tokens (truncated for generation):", n_long_prompts, flush=True)
gens = []
for i in range(0, len(prompts), 8):
    batch = tok(prompts[i:i+8], return_tensors="pt", padding=True, truncation=True,
                max_length=512, add_special_tokens=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=256, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=tok.pad_token_id)
    gens += tok.batch_decode(out[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"generated {len(gens)}/{len(prompts)}", flush=True)
gens = [g.strip() for g in gens]
with open(f"{OUT_DIR}/generations_raw.jsonl", "w") as f:       # saved at once, so a later crash cannot lose them
    for i, g in enumerate(gens):
        f.write(json.dumps({"id": i, "generation": g}, ensure_ascii=False) + "\n")

# ---- 2. NLL of the reference response given the prompt (response tokens only, EOS excluded) ----
NLL_CAP = 1000   # max prompt+response tokens scored; Unsloth inference breaks above 1,024 tokens
nll_nats, ntok, nbytes, n_cut = [], [], [], 0
for r, p in zip(rows, prompts):
    p_ids = tok(p, add_special_tokens=True)["input_ids"][:512]
    r_ids = tok(r["output"], add_special_tokens=False)["input_ids"]
    if len(p_ids) + len(r_ids) > NLL_CAP:                          # score only the part that fits; count its bytes
        r_ids = r_ids[:NLL_CAP - len(p_ids)]; n_cut += 1
        nb = len(tok.decode(r_ids, skip_special_tokens=True).encode("utf-8"))
    else:
        nb = len(r["output"].encode("utf-8"))
    ids = torch.tensor([p_ids + r_ids]).to("cuda")
    with torch.no_grad():
        logits = model(input_ids=ids).logits[0, :-1].float()
    lp = torch.log_softmax(logits, -1)
    tgt = ids[0, 1:]
    tok_lp = lp[torch.arange(len(tgt)), tgt][len(p_ids) - 1:]      # response positions only
    nll_nats.append(-tok_lp.sum().item()); ntok.append(len(r_ids))
    nbytes.append(nb)

def lm_stats(idx):
    n = sum(nll_nats[i] for i in idx)
    return {"bits_per_byte": n / math.log(2) / sum(nbytes[i] for i in idx),
            "token_ppl": math.exp(n / sum(ntok[i] for i in idx))}

# ---- 3. generation metrics ----
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bscore
refs = [r["output"] for r in rows]
rs = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
rouge_items = [rs.score(a, b) for a, b in zip(refs, gens)]
_, _, f1 = bscore(gens, refs, model_type="bert-base-multilingual-cased", rescale_with_baseline=False)
bert_items = f1.numpy()

def corpus_scores(idx):
    h = [gens[i] for i in idx]; r = [refs[i] for i in idx]
    return {"bleu": sacrebleu.corpus_bleu(h, [r]).score,
            "chrfpp": sacrebleu.corpus_chrf(h, [r], word_order=2).score,
            "rouge1": float(np.mean([rouge_items[i]["rouge1"].fmeasure for i in idx])),
            "rouge2": float(np.mean([rouge_items[i]["rouge2"].fmeasure for i in idx])),
            "rougeL": float(np.mean([rouge_items[i]["rougeL"].fmeasure for i in idx])),
            "bertscore_f1": float(np.mean([bert_items[i] for i in idx])),
            **lm_stats(idx)}

def with_ci(idx, B=200):
    rng = random.Random(0)
    base = corpus_scores(idx)
    boots = [corpus_scores([rng.choice(idx) for _ in idx]) for _ in range(B)]
    return {k: {"value": v, "ci95": [float(np.percentile([b[k] for b in boots], 2.5)),
                                     float(np.percentile([b[k] for b in boots], 97.5))]}
            for k, v in base.items()}

# script check: share of outputs containing Devanagari characters
dev = [bool(re.search(r"[ऀ-ॿ]", g)) for g in gens]
groups = {"all": list(range(len(rows)))}
for t in sorted({r["type"] for r in rows}):
    groups[f"type_{t}"] = [i for i, r in enumerate(rows) if r["type"] == t]
import hashlib, importlib.metadata as im
try:
    script_sha = hashlib.sha256(open(__file__, "rb").read()).hexdigest()
except Exception:
    script_sha = "unknown (compute sha256sum eval.py by hand)"
metrics = {"mode": MODE, "model": MODEL_ID, "n": len(rows), "script_sha256": script_sha,
           "package_versions": {p: im.version(p) for p in ["unsloth", "transformers", "peft", "sacrebleu", "bert-score", "torch"]},
           "nll_items_truncated": n_cut, "prompts_over_512_tokens": n_long_prompts,
           "share_outputs_with_devanagari": float(np.mean(dev)),
           "scores": {g: with_ci(ix) for g, ix in groups.items()}}
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1)
with open(f"{OUT_DIR}/outputs.jsonl", "w") as f:
    for i, r in enumerate(rows):
        f.write(json.dumps({"id": i, "type": r["type"], "prompt": prompts[i], "reference": refs[i],
                            "generation": gens[i], "nll_nats": nll_nats[i],
                            "resp_tokens": ntok[i], "resp_bytes": nbytes[i]}, ensure_ascii=False) + "\n")
print(json.dumps({g: {k: round(v["value"], 4) for k, v in s.items()} for g, s in metrics["scores"].items()}, indent=1))

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
model.max_seq_length: 4096
prompts longer than 512 tokens (truncated for generation): 0


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


generated 8/1000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (246 outputs left out)

generated 1000/1000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
 "all": {
  "bleu": 5.6071,
  "chrfpp": 29.5308,
  "rouge1": 0.2503,
  "rouge2": 0.0809,
  "rougeL": 0.2022,
  "bertscore_f1": 0.7557,
  "bits_per_byte": 0.8208,
  "token_ppl": 4.4685
 },
 "type_A": {
  "bleu": 5.4399,
  "chrfpp": 30.7033,
  "rouge1": 0.2564,
  "rouge2": 0.0806,
  "rougeL": 0.2063,
  "bertscore_f1": 0.7595,
  "bits_per_byte": 0.8117,
  "token_ppl": 4.3983
 },
 "type_B": {
  "bleu": 5.7253,
  "chrfpp": 28.3915,
  "rouge1": 0.2441,
  "rouge2": 0.0812,
  "rougeL": 0.198,
  "bertscore_f1": 0.752,
  "bits_per_byte": 0.8296,
  "token_ppl": 4.5374
 }
}


cannonical train.py

In [3]:
%%writefile /kaggle/working/train_canonical.py
# train.py - QLoRA fine-tuning for the Romanized Nepali rerun (identical for every model and seed).
# Run in a Kaggle notebook (GPU T4 x2, Internet on):
#   then edit the CONFIG block below and run this file (%run train.py).
# Set GATE = True first: 50 steps, prints the checks, then stop and read the output.
import json, math, os, time
import torch

# ---------------- CONFIG (the only block you edit) ----------------
MODEL_ID = "Qwen/Qwen2.5-3B"     # google/gemma-2-2b | Qwen/Qwen2.5-3B | meta-llama/Llama-3.2-3B
SEED = 42                                # 42, 43, 44
DATA_DIR = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected"  # holds train.jsonl and validation.jsonl
OUT_DIR = "/kaggle/working/run"          # adapter, logs and training_meta.json are written here
GATE = False                              # True = 50-step check only
# ------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

ALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "
          "Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
ALPACA_NO_INPUT = ("Below is an instruction that describes a task. "
                   "Write a response that appropriately completes the request.\n\n"
                   "### Instruction:\n{instruction}\n\n### Response:\n")

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=512, dtype=torch.float16, load_in_4bit=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0.0, bias="none", use_rslora=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"TRAINABLE PARAMETERS: {n_train:,} (counted on the 4-bit model; {n_all:,} total as stored)")

def fmt(ex):
    p = (ALPACA if ex["input"].strip() else ALPACA_NO_INPUT).format(**ex)
    return {"text": p + ex["output"] + tok.eos_token}

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                        "validation": f"{DATA_DIR}/validation.jsonl"})
data = data.map(fmt)

# --- sanity checks that must be read in the gate run ---
first = tok(data["train"][0]["text"])["input_ids"]
print("BOS id:", tok.bos_token_id, "| first token id:", first[0],
      "| starts with BOS:", (first[0] == tok.bos_token_id) if tok.bos_token_id is not None else "n/a (no BOS)")
print("Last 3 token ids of first example:", first[-3:], "| EOS id:", tok.eos_token_id,
      "(should end with exactly one EOS)")
print("Vocab size:", len(tok), "| uniform-guess loss ln(V) = %.2f" % math.log(len(tok)))

cfg_kwargs = dict(
    output_dir=OUT_DIR, dataset_text_field="text", packing=False,
    per_device_train_batch_size=2, gradient_accumulation_steps=4, per_device_eval_batch_size=2,
    num_train_epochs=3, max_steps=50 if GATE else -1,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=200, optim="adamw_8bit",
    fp16=True, bf16=False, seed=SEED, data_seed=SEED,
    eval_strategy="steps", eval_steps=100 if not GATE else 25,
    save_strategy="steps", save_steps=100 if not GATE else 25, save_total_limit=10,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, report_to="none")
# TRL renamed max_seq_length -> max_length and tokenizer -> processing_class; try the new name first.
try:
    cfg = SFTConfig(max_length=512, **cfg_kwargs)
except TypeError:
    cfg = SFTConfig(max_seq_length=512, **cfg_kwargs)
try:
    trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=data["train"],
                         eval_dataset=data["validation"], args=cfg)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=data["train"],
                         eval_dataset=data["validation"], args=cfg)

step0 = trainer.evaluate()["eval_loss"]
print(f"STEP-0 EVAL LOSS: {step0:.3f}  (must be well below ln(V) above; the old Gemma run showed 22.6)")

t0 = time.time()
trainer.train()
secs = time.time() - t0
steps = trainer.state.global_step
print(f"SECONDS PER STEP: {secs/steps:.2f}")

trainer.model.save_pretrained(f"{OUT_DIR}/adapter")
tok.save_pretrained(f"{OUT_DIR}/adapter")
import hashlib, importlib.metadata as im
try:
    script_sha = hashlib.sha256(open(__file__, "rb").read()).hexdigest()
except Exception:
    script_sha = "unknown (compute sha256sum train.py by hand)"
versions = {p: im.version(p) for p in ["unsloth", "trl", "transformers", "peft", "bitsandbytes", "torch"]}
meta = {"model": MODEL_ID, "seed": SEED, "gate": GATE, "trainable_params": n_train,
        "script_sha256": script_sha, "package_versions": versions,
        "effective_batch": cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps
                           * max(trainer.args.world_size, 1),
        "num_gpus_visible": torch.cuda.device_count(), "world_size": trainer.args.world_size,
        "global_steps": steps, "seconds_per_step": secs / steps, "step0_eval_loss": step0,
        "best_step": trainer.state.best_global_step, "best_eval_loss": trainer.state.best_metric,
        "log_history": trainer.state.log_history}
json.dump(meta, open(f"{OUT_DIR}/training_meta.json", "w"), indent=1)
print("DONE. Best step:", meta["best_step"], "best val loss:", meta["best_eval_loss"],
      "| steps:", steps, "| effective batch:", meta["effective_batch"], "| world size:", meta["world_size"])

Writing /kaggle/working/train_canonical.py


parallel train SEED 43 44

In [4]:
# parallel_train.py - runs the UNCHANGED train.py twice at the same time, one seed per GPU.
# How: for each job it writes a copy of train.py in which only the CONFIG lines listed below differ
# (the script proves this with a line diff), then runs each copy with CUDA_VISIBLE_DEVICES set to one GPU.
# Training settings, seeds and data are untouched. Run:  !python parallel_train.py   (GPU T4 x2, Internet on)

# ---------------- EDIT ONLY THIS BLOCK ----------------
TRAIN_PY = "/kaggle/working/train_canonical.py"          # the canonical train.py (do not edit it)
DATA_DIR = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected"         # folder with train.jsonl and validation.jsonl
MODEL_ID = "Qwen/Qwen2.5-3B"
WORKDIR = "/kaggle/working/launch"             # patched copies, logs and launch records go here
STAGGER_SECONDS = 90
JOBS = [
    {"tag": "qwen_s43", "gpu": 0, "set": {"MODEL_ID": MODEL_ID, "SEED": 43, "DATA_DIR": DATA_DIR,
                                           "OUT_DIR": "/kaggle/working/qwen_s43", "GATE": False}},
    {"tag": "qwen_s44", "gpu": 1, "set": {"MODEL_ID": MODEL_ID, "SEED": 44, "DATA_DIR": DATA_DIR,
                                           "OUT_DIR": "/kaggle/working/qwen_s44", "GATE": False}},
]
# ------------------------------------------------------

import ast, difflib, hashlib, json, os, re, subprocess, sys, time

def _sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

def patch_script(script, overrides, dest):
    """Copy `script` to `dest`, changing ONLY the CONFIG lines named in `overrides`."""
    lines = open(script, encoding="utf-8").read().split("\n")
    start = next(i for i, l in enumerate(lines) if l.startswith("# ---") and "CONFIG" in l)
    end = next(i for i in range(start + 1, len(lines)) if lines[i].startswith("# ---"))
    changed = set()
    for i in range(start + 1, end):
        m = re.match(r"^([A-Z_]+)\s*=", lines[i])
        if m and m.group(1) in overrides:
            lines[i] = f"{m.group(1)} = {overrides[m.group(1)]!r}"
            changed.add(m.group(1))
    missing = set(overrides) - changed
    assert not missing, f"CONFIG keys not found in {script}: {missing}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest, "w", encoding="utf-8").write("\n".join(lines))
    # safety proof: the copy may differ from the original in exactly the overridden lines
    a = open(script, encoding="utf-8").read().split("\n")
    n_diff = sum(1 for d in difflib.ndiff(a, lines) if d.startswith("- "))
    assert n_diff <= len(overrides), "patched copy differs in more lines than expected"
    return dest

def run_jobs(script, jobs, workdir, stagger):
    os.makedirs(workdir, exist_ok=True)
    try:
        n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    except Exception:
        n_gpu = 0
    print(f"GPUs visible: {n_gpu}")
    assert n_gpu >= len(jobs), "need one GPU per job (turn on GPU T4 x2)"
    pristine = _sha(script)
    print("pristine script sha256:", pristine)
    procs = []
    for k, job in enumerate(jobs):
        tag = job["tag"]
        patched = patch_script(script, job["set"], f"{workdir}/{tag}_{os.path.basename(script)}")
        log = open(f"{workdir}/{tag}.log", "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(job["gpu"]), PYTHONUNBUFFERED="1")
        rec = {"tag": tag, "gpu": job["gpu"], "overrides": job["set"], "pristine_sha256": pristine,
               "patched_sha256": _sha(patched), "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
        json.dump(rec, open(f"{workdir}/{tag}_launch.json", "w"), indent=1)
        if os.environ.get("LAUNCH_DRY"):
            print("DRY RUN, patched copy written:", patched); continue
        if k > 0:
            print(f"waiting {stagger}s before starting the next job (avoids two simultaneous downloads)")
            time.sleep(stagger)
        print(f"starting {tag} on GPU {job['gpu']}")
        procs.append((tag, subprocess.Popen([sys.executable, patched], stdout=log, stderr=subprocess.STDOUT, env=env), log))
    last = 0
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(20)
        if time.time() - last > 300:
            last = time.time()
            print(time.strftime("[%H:%M:%S]"))
            for tag, p, _ in procs:
                txt = open(f"{workdir}/{tag}.log", errors="ignore").read()[-400:].replace("\r", "\n").strip().splitlines()
                print(f"  {tag} ({'running' if p.poll() is None else 'exit ' + str(p.returncode)}):", txt[-1][:150] if txt else "")
    ok = True
    for tag, p, log in procs:
        log.close()
        print(f"{tag}: exit code {p.returncode}")
        ok &= (p.returncode == 0)
        tail = open(f"{workdir}/{tag}.log", errors="ignore").read().replace("\r", "\n").strip().splitlines()[-12:]
        print("\n".join("   " + t[:200] for t in tail))
    print("ALL JOBS FINISHED OK" if ok and procs else "SOMETHING FAILED OR DRY RUN: read the tails above")

run_jobs(TRAIN_PY, JOBS, WORKDIR, STAGGER_SECONDS)

GPUs visible: 2
pristine script sha256: d9d03919032bfe809bd868bbf2a99720423066d3085e82d1c3cf18562e0a2d12
starting qwen_s43 on GPU 0
waiting 90s before starting the next job (avoids two simultaneous downloads)
starting qwen_s44 on GPU 1
[03:59:02]
  qwen_s43 (running):  76%|███████▌  | 189/250 [00:50<00:15,  3.87it/s]
  qwen_s44 (running): Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[04:04:02]
  qwen_s44 (running):   3%|▎         | 85/3375 [03:22<2:10:46,  2.39s/it]
[04:09:02]
  qwen_s44 (running):   6%|▌         | 186/3375 [08:22<2:00:27,  2.27s/it]
[04:14:02]
  qwen_s44 (running):   9%|▊         | 288/3375 [13:21<1:56:54,  2.27s/it]
[04:19:02]
  qwen_s44 (running):  11%|█▏        | 388/3375 [18:21<2:02:03,  2.45s/it]
[04:24:02]
  qwen_s44 (running):  15%|█▍        | 490/3375 [23:21<1:53:14,  2.35s/it]
[04:29:02]
  qwen_s44 (running):  18%|█▊        | 593/3375 [28:22<1:55:28,  2.49s/it]
[04:34:02]
  qwen_s44 (running):  21%|██        | 694/3375

In [10]:
%%writefile /kaggle/working/eval.py
# eval.py v2.2 (adds scoring cap, early save of generations, per-item NLL) - one evaluation script for zero-shot and fine-tuned models (identical for every model).
# Run in a Kaggle notebook (GPU, Internet on):

# Edit CONFIG, then %run eval.py. Writes outputs.jsonl and metrics.json into OUT_DIR.
import json, math, os, random, re
import numpy as np
import torch

# ---------------- CONFIG ----------------
MODE = "finetuned"                       # "zeroshot" (bare instruction, base weights) or "finetuned"
MODEL_ID = "Qwen/Qwen2.5-3B"     # base model id (used for zeroshot)
ADAPTER_DIR = "/kaggle/input/datasets/pujanpant017/adaptaer-all/adapter_001/adapter_001"   # used when MODE == "finetuned"
DATA_FILE = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected/test.jsonl"
OUT_DIR = "/kaggle/working/eval_seed42"
N_ITEMS = None                           # None = all 1,000; set e.g. 500 only if the pilot says time is short
# ----------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
from unsloth import FastLanguageModel

ALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "
          "Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
ALPACA_NO_INPUT = ("Below is an instruction that describes a task. "
                   "Write a response that appropriately completes the request.\n\n"
                   "### Instruction:\n{instruction}\n\n### Response:\n")

def build_prompt(r):
    if MODE == "zeroshot":   # Rimal and Rimal: no system prompt, template or few-shot examples
        return r["instruction"] + ("\n" + r["input"] if r["input"].strip() else "")
    return (ALPACA if r["input"].strip() else ALPACA_NO_INPUT).format(**r)

rows = [json.loads(l) for l in open(DATA_FILE)]
if N_ITEMS:
    random.Random(42).shuffle(rows); rows = rows[:N_ITEMS]

load_name = ADAPTER_DIR if MODE == "finetuned" else MODEL_ID   # adapter on the 4-bit base, never merged
model, tok = FastLanguageModel.from_pretrained(model_name=load_name, max_seq_length=4096,
                                               dtype=torch.float16, load_in_4bit=True)
FastLanguageModel.for_inference(model)
print("model.max_seq_length:", getattr(model, "max_seq_length", "n/a"), flush=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

# ---- 1. greedy generation, identical for both stages ----
prompts = [build_prompt(r) for r in rows]
n_long_prompts = sum(len(tok(p, add_special_tokens=True)["input_ids"]) > 512 for p in prompts)
print("prompts longer than 512 tokens (truncated for generation):", n_long_prompts, flush=True)
gens = []
for i in range(0, len(prompts), 8):
    batch = tok(prompts[i:i+8], return_tensors="pt", padding=True, truncation=True,
                max_length=512, add_special_tokens=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=256, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=tok.pad_token_id)
    gens += tok.batch_decode(out[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"generated {len(gens)}/{len(prompts)}", flush=True)
gens = [g.strip() for g in gens]
with open(f"{OUT_DIR}/generations_raw.jsonl", "w") as f:       # saved at once, so a later crash cannot lose them
    for i, g in enumerate(gens):
        f.write(json.dumps({"id": i, "generation": g}, ensure_ascii=False) + "\n")

# ---- 2. NLL of the reference response given the prompt (response tokens only, EOS excluded) ----
NLL_CAP = 1000   # max prompt+response tokens scored; Unsloth inference breaks above 1,024 tokens
nll_nats, ntok, nbytes, n_cut = [], [], [], 0
for r, p in zip(rows, prompts):
    p_ids = tok(p, add_special_tokens=True)["input_ids"][:512]
    r_ids = tok(r["output"], add_special_tokens=False)["input_ids"]
    if len(p_ids) + len(r_ids) > NLL_CAP:                          # score only the part that fits; count its bytes
        r_ids = r_ids[:NLL_CAP - len(p_ids)]; n_cut += 1
        nb = len(tok.decode(r_ids, skip_special_tokens=True).encode("utf-8"))
    else:
        nb = len(r["output"].encode("utf-8"))
    ids = torch.tensor([p_ids + r_ids]).to("cuda")
    with torch.no_grad():
        logits = model(input_ids=ids).logits[0, :-1].float()
    lp = torch.log_softmax(logits, -1)
    tgt = ids[0, 1:]
    tok_lp = lp[torch.arange(len(tgt)), tgt][len(p_ids) - 1:]      # response positions only
    nll_nats.append(-tok_lp.sum().item()); ntok.append(len(r_ids))
    nbytes.append(nb)

def lm_stats(idx):
    n = sum(nll_nats[i] for i in idx)
    return {"bits_per_byte": n / math.log(2) / sum(nbytes[i] for i in idx),
            "token_ppl": math.exp(n / sum(ntok[i] for i in idx))}

# ---- 3. generation metrics ----
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bscore
refs = [r["output"] for r in rows]
rs = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
rouge_items = [rs.score(a, b) for a, b in zip(refs, gens)]
_, _, f1 = bscore(gens, refs, model_type="bert-base-multilingual-cased", rescale_with_baseline=False)
bert_items = f1.numpy()

def corpus_scores(idx):
    h = [gens[i] for i in idx]; r = [refs[i] for i in idx]
    return {"bleu": sacrebleu.corpus_bleu(h, [r]).score,
            "chrfpp": sacrebleu.corpus_chrf(h, [r], word_order=2).score,
            "rouge1": float(np.mean([rouge_items[i]["rouge1"].fmeasure for i in idx])),
            "rouge2": float(np.mean([rouge_items[i]["rouge2"].fmeasure for i in idx])),
            "rougeL": float(np.mean([rouge_items[i]["rougeL"].fmeasure for i in idx])),
            "bertscore_f1": float(np.mean([bert_items[i] for i in idx])),
            **lm_stats(idx)}

def with_ci(idx, B=200):
    rng = random.Random(0)
    base = corpus_scores(idx)
    boots = [corpus_scores([rng.choice(idx) for _ in idx]) for _ in range(B)]
    return {k: {"value": v, "ci95": [float(np.percentile([b[k] for b in boots], 2.5)),
                                     float(np.percentile([b[k] for b in boots], 97.5))]}
            for k, v in base.items()}

# script check: share of outputs containing Devanagari characters
dev = [bool(re.search(r"[ऀ-ॿ]", g)) for g in gens]
groups = {"all": list(range(len(rows)))}
for t in sorted({r["type"] for r in rows}):
    groups[f"type_{t}"] = [i for i, r in enumerate(rows) if r["type"] == t]
import hashlib, importlib.metadata as im
try:
    script_sha = hashlib.sha256(open(__file__, "rb").read()).hexdigest()
except Exception:
    script_sha = "unknown (compute sha256sum eval.py by hand)"
metrics = {"mode": MODE, "model": MODEL_ID, "n": len(rows), "script_sha256": script_sha,
           "package_versions": {p: im.version(p) for p in ["unsloth", "transformers", "peft", "sacrebleu", "bert-score", "torch"]},
           "nll_items_truncated": n_cut, "prompts_over_512_tokens": n_long_prompts,
           "share_outputs_with_devanagari": float(np.mean(dev)),
           "scores": {g: with_ci(ix) for g, ix in groups.items()}}
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1)
with open(f"{OUT_DIR}/outputs.jsonl", "w") as f:
    for i, r in enumerate(rows):
        f.write(json.dumps({"id": i, "type": r["type"], "prompt": prompts[i], "reference": refs[i],
                            "generation": gens[i], "nll_nats": nll_nats[i],
                            "resp_tokens": ntok[i], "resp_bytes": nbytes[i]}, ensure_ascii=False) + "\n")
print(json.dumps({g: {k: round(v["value"], 4) for k, v in s.items()} for g, s in metrics["scores"].items()}, indent=1))

Overwriting /kaggle/working/eval.py


PARALLEL EVAL SEED 43 44

In [11]:
# parallel_eval.py - runs the UNCHANGED eval.py twice at the same time, one adapter per GPU.
# Same mechanism as parallel_train.py (only the CONFIG lines listed below differ; proven by a line diff).
# Run:  !python parallel_eval.py   (GPU T4 x2, Internet on)

# ---------------- EDIT ONLY THIS BLOCK ----------------
EVAL_PY = "/kaggle/working/eval.py"            # the canonical eval.py (do not edit it)
TEST_FILE = "/kaggle/input/datasets/pujanpant017/dataste001/dataset_corrected/test.jsonl"
MODEL_ID = "Qwen/Qwen2.5-3B"
WORKDIR = "/kaggle/working/launch"
STAGGER_SECONDS = 90
JOBS = [
    {"tag": "eval_qwen_s43", "gpu": 0, "set": {"MODE": "finetuned", "MODEL_ID": MODEL_ID,
        "ADAPTER_DIR": "/kaggle/input/datasets/pujanpant017/adapter4344/adapter43/adapter43", "DATA_FILE": TEST_FILE,
        "OUT_DIR": "/kaggle/working/eval_qwen_s43", "N_ITEMS": None}},
    {"tag": "eval_qwen_s44", "gpu": 1, "set": {"MODE": "finetuned", "MODEL_ID": MODEL_ID,
        "ADAPTER_DIR": "/kaggle/input/datasets/pujanpant017/adapter4344/adapter44/adapter44", "DATA_FILE": TEST_FILE,
        "OUT_DIR": "/kaggle/working/eval_qwen_s44", "N_ITEMS": None}},
]
# ------------------------------------------------------

import ast, difflib, hashlib, json, os, re, subprocess, sys, time

def _sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

def patch_script(script, overrides, dest):
    """Copy `script` to `dest`, changing ONLY the CONFIG lines named in `overrides`."""
    lines = open(script, encoding="utf-8").read().split("\n")
    start = next(i for i, l in enumerate(lines) if l.startswith("# ---") and "CONFIG" in l)
    end = next(i for i in range(start + 1, len(lines)) if lines[i].startswith("# ---"))
    changed = set()
    for i in range(start + 1, end):
        m = re.match(r"^([A-Z_]+)\s*=", lines[i])
        if m and m.group(1) in overrides:
            lines[i] = f"{m.group(1)} = {overrides[m.group(1)]!r}"
            changed.add(m.group(1))
    missing = set(overrides) - changed
    assert not missing, f"CONFIG keys not found in {script}: {missing}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest, "w", encoding="utf-8").write("\n".join(lines))
    # safety proof: the copy may differ from the original in exactly the overridden lines
    a = open(script, encoding="utf-8").read().split("\n")
    n_diff = sum(1 for d in difflib.ndiff(a, lines) if d.startswith("- "))
    assert n_diff <= len(overrides), "patched copy differs in more lines than expected"
    return dest

def run_jobs(script, jobs, workdir, stagger):
    os.makedirs(workdir, exist_ok=True)
    try:
        n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    except Exception:
        n_gpu = 0
    print(f"GPUs visible: {n_gpu}")
    assert n_gpu >= len(jobs), "need one GPU per job (turn on GPU T4 x2)"
    pristine = _sha(script)
    print("pristine script sha256:", pristine)
    procs = []
    for k, job in enumerate(jobs):
        tag = job["tag"]
        patched = patch_script(script, job["set"], f"{workdir}/{tag}_{os.path.basename(script)}")
        log = open(f"{workdir}/{tag}.log", "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(job["gpu"]), PYTHONUNBUFFERED="1")
        rec = {"tag": tag, "gpu": job["gpu"], "overrides": job["set"], "pristine_sha256": pristine,
               "patched_sha256": _sha(patched), "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
        json.dump(rec, open(f"{workdir}/{tag}_launch.json", "w"), indent=1)
        if os.environ.get("LAUNCH_DRY"):
            print("DRY RUN, patched copy written:", patched); continue
        if k > 0:
            print(f"waiting {stagger}s before starting the next job (avoids two simultaneous downloads)")
            time.sleep(stagger)
        print(f"starting {tag} on GPU {job['gpu']}")
        procs.append((tag, subprocess.Popen([sys.executable, patched], stdout=log, stderr=subprocess.STDOUT, env=env), log))
    last = 0
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(20)
        if time.time() - last > 300:
            last = time.time()
            print(time.strftime("[%H:%M:%S]"))
            for tag, p, _ in procs:
                txt = open(f"{workdir}/{tag}.log", errors="ignore").read()[-400:].replace("\r", "\n").strip().splitlines()
                print(f"  {tag} ({'running' if p.poll() is None else 'exit ' + str(p.returncode)}):", txt[-1][:150] if txt else "")
    ok = True
    for tag, p, log in procs:
        log.close()
        print(f"{tag}: exit code {p.returncode}")
        ok &= (p.returncode == 0)
        tail = open(f"{workdir}/{tag}.log", errors="ignore").read().replace("\r", "\n").strip().splitlines()[-12:]
        print("\n".join("   " + t[:200] for t in tail))
    print("ALL JOBS FINISHED OK" if ok and procs else "SOMETHING FAILED OR DRY RUN: read the tails above")

run_jobs(EVAL_PY, JOBS, WORKDIR, STAGGER_SECONDS)

GPUs visible: 2
pristine script sha256: a231a1ae64bc0b1784d7b16913d92dfc9196c15ca609e8c2bc2455c062c1672f
starting eval_qwen_s43 on GPU 0
waiting 90s before starting the next job (avoids two simultaneous downloads)
starting eval_qwen_s44 on GPU 1
[08:38:57]
  eval_qwen_s43 (running): Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation 
  eval_qwen_s44 (running): Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[08:43:57]
  eval_qwen_s43 (running): Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation 
  eval_qwen_s44 (running): Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation 
[08:48:57]
  eval_qwen_s43 (running): Both `max_new_tokens` (=256) and `max_length`(=